# ACN-Data API Tutorial (Fixed Authentication)

This notebook fetches EV charging sessions from the ACN-Data dataset. It safely prompts for your API token and flattens all nested JSON attributes into a complete Pandas DataFrame.

In [ ]:
# 1. Install the necessary libraries
!pip install acnportal pandas

### 2. Enter API Token and Initialize Client
When you run this cell, an input box will appear. Paste your API token from `https://ev.caltech.edu/dataset`. **Do not use your email or password!**

In [ ]:
import getpass
from acnportal import acndata
import pandas as pd
from datetime import datetime
import json

print("Please enter your ACN-Data API Token.")
# getpass creates a secure input box so you don't accidentally hardcode a bad string
API_TOKEN = getpass.getpass("API Token: ")

# Initialize the DataClient
client = acndata.DataClient(API_TOKEN)

### 3. Fetch Data and Flatten Attributes
We will query for sessions at Caltech for the first week of September 2019. The script will use `pd.json_normalize()` to ensure that all nested JSON elements (like `userInputs`) are properly separated into individual columns.

In [ ]:
start_time = datetime(2019, 9, 1)
end_time = datetime(2019, 9, 7)

try:
    print("Fetching data... (The client automatically handles HATEOAS pagination links!)")
    
    # Fetch sessions (this returns a generator)
    sessions_generator = client.get_sessions_by_time(
        site="caltech", 
        start=start_time, 
        end=end_time,
        min_energy=10.0
    )

    # Convert generator directly into a list of dictionaries
    session_list = list(sessions_generator)

    if len(session_list) == 0:
        print("No sessions found for this specific time frame and criteria.")
    else:
        # pd.json_normalize perfectly flattens nested dictionaries.
        # For example, "userInputs": {"WhPerMile": 400} becomes the column "userInputs.WhPerMile"
        df = pd.json_normalize(session_list)
        
        # Convert time strings to proper datetime objects
        time_columns = ["connectionTime", "disconnectTime", "doneChargingTime"]
        for col in time_columns:
            if col in df.columns:
                df[col] = pd.to_datetime(df[col])

        print(f"\n✅ Success! Downloaded {len(df)} sessions.")
        print(f"✅ Extracted {len(df.columns)} columns (all nested attributes included).")
        
        # Show all columns without truncation
        pd.set_option('display.max_columns', None)
        display(df.head())
        
except json.JSONDecodeError:
    print("\n🚨 AUTHENTICATION ERROR: Expecting value: line 1 column 1 (char 0)")
    print("The server rejected your API token. Please ensure you pasted the exact 64-character token, NOT your registration email or password.")
except Exception as e:
    print(f"\n🚨 An unexpected error occurred: {e}")